# Recovery Notebook - Save & Test Fine-Tuned Model

This notebook loads your **already-trained checkpoint** from Google Drive and:
1. Saves the LoRA adapters
2. Exports a GGUF model for Ollama
3. Tests the model

**This is fast** - should take only ~10-15 minutes total.

## Step 1: Install Dependencies (~3 min)

In [ ]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

## Step 2: Mount Google Drive & Find Checkpoint

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob

# Look for checkpoints
checkpoint_dir = '/content/drive/MyDrive/cybersec-llama2-finetuned'

if os.path.exists(checkpoint_dir):
    items = sorted(os.listdir(checkpoint_dir))
    checkpoints = [d for d in items if d.startswith('checkpoint-')]
    print(f'Found {len(checkpoints)} checkpoint(s):')
    for cp in checkpoints:
        cp_path = os.path.join(checkpoint_dir, cp)
        size = sum(os.path.getsize(os.path.join(cp_path, f)) for f in os.listdir(cp_path) if os.path.isfile(os.path.join(cp_path, f)))
        print(f'  - {cp} ({size/1024/1024:.1f} MB)')
    if checkpoints:
        CHECKPOINT_PATH = os.path.join(checkpoint_dir, checkpoints[-1])
        print(f'\nUsing latest: {CHECKPOINT_PATH}')
    else:
        print('No checkpoints found! Check if training saved any files.')
        CHECKPOINT_PATH = None
else:
    print(f'Directory not found: {checkpoint_dir}')
    print('Check your Google Drive for the training output folder.')
    CHECKPOINT_PATH = None

## Step 3: Load Base Model + Trained Checkpoint
We load the original Llama-2-7B and then apply your trained LoRA weights on top.

In [ ]:
from unsloth import FastLanguageModel
import torch

# Load the base model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='unsloth/llama-2-7b-bnb-4bit',
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)
print('Base model loaded.')

# Now load the trained LoRA adapter from checkpoint
from peft import PeftModel
model = PeftModel.from_pretrained(model, CHECKPOINT_PATH)
print(f'Trained LoRA adapter loaded from: {CHECKPOINT_PATH}')
print('Model is ready!')

## Step 4: Save LoRA Adapters (Step 9)

In [ ]:
LORA_OUTPUT = '/content/drive/MyDrive/cybersec-llama2-lora'
model.save_pretrained(LORA_OUTPUT)
tokenizer.save_pretrained(LORA_OUTPUT)
print(f'LoRA adapters saved to: {LORA_OUTPUT}')

## Step 5: Export GGUF Model for Ollama (Step 9)
This converts the model to GGUF format so you can use it with Ollama on your PC.

**This step takes ~5-10 minutes.**

In [ ]:
GGUF_OUTPUT = '/content/drive/MyDrive/cybersec-llama2-gguf'

# We need to merge LoRA into base model first for GGUF export
# Reload with Unsloth's method for proper GGUF support
from unsloth import FastLanguageModel

model2, tokenizer2 = FastLanguageModel.from_pretrained(
    model_name=CHECKPOINT_PATH,
    max_seq_length=4096,
    dtype=None,
    load_in_4bit=True,
)

model2.save_pretrained_gguf(
    GGUF_OUTPUT,
    tokenizer2,
    quantization_method='q4_k_m',
)
print(f'GGUF model saved to: {GGUF_OUTPUT}')
print('Format: Q4_K_M (recommended for Ollama)')

# Show file size
import os
for f in os.listdir(GGUF_OUTPUT):
    fpath = os.path.join(GGUF_OUTPUT, f)
    if os.path.isfile(fpath):
        print(f'  {f}: {os.path.getsize(fpath)/1024/1024/1024:.2f} GB')

## Step 6: Test the Fine-Tuned Model (Step 10)

In [ ]:
FastLanguageModel.for_inference(model2)

test_question = 'How would you detect lateral movement using Windows Event ID correlation in an enterprise environment?'

prompt = (
    '<s>[INST] <<SYS>>\n'
    'You are an advanced AI assistant specialized in cybersecurity causal reasoning and threat analysis.\n'
    '<</SYS>>\n\n'
    f'{test_question} [/INST] '
)

inputs = tokenizer2(prompt, return_tensors='pt').to('cuda')

outputs = model2.generate(
    **inputs,
    max_new_tokens=512,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.1,
)

response = tokenizer2.decode(outputs[0], skip_special_tokens=True)

print('=' * 60)
print('CYBERSECURITY AI ASSISTANT - TEST')
print('=' * 60)
print(f'\nQuestion: {test_question}')
print(f'\nResponse:\n{response.split("[/INST]")[-1].strip()}')

## Done!

Your model files are saved in Google Drive:

| Folder | Use |
|--------|-----|
| `cybersec-llama2-lora/` | LoRA adapters (for further training) |
| `cybersec-llama2-gguf/` | GGUF file (for Ollama on your PC) |

### To use with Ollama:
1. Download `cybersec-llama2-gguf/` folder to your PC
2. Create a `Modelfile`:
```
FROM ./unsloth.Q4_K_M.gguf
SYSTEM "You are an advanced AI assistant specialized in cybersecurity..."
```
3. Run: `ollama create cybersec-assistant -f Modelfile`
4. Chat: `ollama run cybersec-assistant`